In [1]:
import numpy as np

class GridWorld:
    def __init__(self, size, goal, trap):
        self.size = size
        self.goal = goal
        self.trap = trap
        self.actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']
        self.states = [(i, j) for i in range(size) for j in range(size)]

    def step(self, state, action):
        i, j = state

        if action == 'UP': i -= 1
        elif action == 'DOWN': i += 1
        elif action == 'LEFT': j -= 1
        elif action == 'RIGHT': j += 1

        i = max(0, min(i, self.size - 1))
        j = max(0, min(j, self.size - 1))
        next_state = (i, j)

        if next_state == self.goal:
            return next_state, 0
        elif next_state == self.trap:
            return next_state, -10
        else:
            return next_state, -1


def policy_iteration(env, gamma=0.9):
    policy = {s: np.random.choice(env.actions) for s in env.states}
    V = {s: 0 for s in env.states}

    while True:
        # Policy Evaluation
        while True:
            delta = 0
            for s in env.states:
                if s == env.goal or s == env.trap:
                    continue

                v = V[s]
                a = policy[s]
                ns, r = env.step(s, a)
                V[s] = r + gamma * V[ns]
                delta = max(delta, abs(v - V[s]))

            if delta < 0.01:
                break

        # Policy Improvement
        stable = True
        for s in env.states:
            if s == env.goal or s == env.trap:
                continue

            old_action = policy[s]

            policy[s] = max(
                env.actions,
                key=lambda a: env.step(s, a)[1] +
                              gamma * V[env.step(s, a)[0]]
            )

            if old_action != policy[s]:
                stable = False

        if stable:
            break

    return policy, V


# Example
env = GridWorld(3, goal=(2, 2), trap=(1, 1))
policy, V = policy_iteration(env)

print("Optimal Policy:")
for s, a in policy.items():
    print(s, ":", a)

Optimal Policy:
(0, 0) : DOWN
(0, 1) : RIGHT
(0, 2) : DOWN
(1, 0) : DOWN
(1, 1) : RIGHT
(1, 2) : DOWN
(2, 0) : RIGHT
(2, 1) : RIGHT
(2, 2) : RIGHT
